# Multiple Linear Regression

### 本册学什么？

**多元线性回归（Multiple Linear Regression）**是简单线性回归的升级版：不再只用一个因素预测结果，而是同时用多个因素。本册用"研发投入、管理费用、营销费用、所在州"这 4 个因素预测公司利润，模型会自动学习每个因素的"影响力"（系数）。

流程和上一册类似：读数据 → 预处理（把"州"这种文字变成数字）→ 分训练/测试集 → 训练（fit）→ 预测（predict）。运行完本册，最后你会看到一张两列的对照表：左列是模型预测的利润，右列是真实利润。

## Importing the libraries

### 准备工作：导入需要的工具库

Python 本身功能很少，做数值计算、画图、读表格都要先"导入"相应的库，相当于做菜前先把锅碗瓢盆摆好。这一步没有任何输出，运行后不报错就说明环境正常。

In [1]:
%matplotlib inline  # 魔法命令（以 % 开头的特殊指令）：让 matplotlib 画的图直接嵌在笔记本页面里显示，而不是弹出独立窗口
import numpy as np  # 导入 numpy 数值计算库，约定俗成简写为 np，专门用来高效处理数组（一串数字）的运算
import matplotlib.pyplot as plt  # 导入 matplotlib 库中的 pyplot 画图模块，简写为 plt，负责把数据画成图表
import pandas as pd  # 导入 pandas 数据处理库，简写为 pd，负责读取和处理表格数据（类似 Excel）

## Importing the dataset

### 读取数据

数据在 50_Startups.csv 里，记录了 50 家公司的研发投入、管理费用、营销费用、所在州和利润。X 存前 4 列（输入特征），y 存最后一列"利润"（要预测的目标）。

In [2]:
# 读取 CSV 数据：50_Startups.csv 记录了 50 家公司的信息
dataset = pd.read_csv('50_Startups.csv')
# iloc 是"按位置取数"：方括号里写成 [行, 列]，逗号前选行、逗号后选列
# 第一个冒号表示"所有行"；:-1 表示"从第 1 列取到倒数第 2 列"（不含最后一列）
# 所以 X 是特征：研发投入、管理费用、营销费用、所在州
X = dataset.iloc[:, :-1].values
# -1 表示"最后一列"。y 是要预测的目标：利润
y = dataset.iloc[:, -1].values

### 先看一眼数据

下面两个单元格不做任何计算，只是"查看"：确认列名和数据长什么样。动手分析前先看一眼数据是好习惯。

In [3]:
# 查看表里所有列的名字，确认每一列是什么含义
dataset.columns

Index(['R&D Spend', 'Administration', 'Marketing Spend', 'State', 'Profit'], dtype='object')

In [4]:
# 单独写一个 X 回车，笔记本会自动显示它的值（这是快速查看变量的技巧）
X

array([[165349.2, 136897.8, 471784.1, 'New York'],
       [162597.7, 151377.59, 443898.53, 'California'],
       [153441.51, 101145.55, 407934.54, 'Florida'],
       [144372.41, 118671.85, 383199.62, 'New York'],
       [142107.34, 91391.77, 366168.42, 'Florida'],
       [131876.9, 99814.71, 362861.36, 'New York'],
       [134615.46, 147198.87, 127716.82, 'California'],
       [130298.13, 145530.06, 323876.68, 'Florida'],
       [120542.52, 148718.95, 311613.29, 'New York'],
       [123334.88, 108679.17, 304981.62, 'California'],
       [101913.08, 110594.11, 229160.95, 'Florida'],
       [100671.96, 91790.61, 249744.55, 'California'],
       [93863.75, 127320.38, 249839.44, 'Florida'],
       [91992.39, 135495.07, 252664.93, 'California'],
       [119943.24, 156547.42, 256512.92, 'Florida'],
       [114523.61, 122616.84, 261776.23, 'New York'],
       [78013.11, 121597.55, 264346.06, 'California'],
       [94657.16, 145077.58, 282574.31, 'New York'],
       [91749.16, 114175.79, 29491

## Encoding categorical data

### 把文字类别变成数字（独热编码）

数据里"所在州"是文字（如 New York / California / Florida），而模型只认识数字。独热编码的做法是：为每个州新建一列，属于该州就填 1、不属于就填 0，比如 1,0,0 代表纽约。注意不能用 1、2、3 给各州编号，因为那样会暗示它们有大小关系，而实际上没有。

In [5]:
# 从 sklearn 导入"列变换器"，专门用来对表格中指定的列做变换
from sklearn.compose import ColumnTransformer
# 导入"独热编码器"：把文字类别变成 0/1 组成的数字，模型才读得懂
from sklearn.preprocessing import OneHotEncoder
# 建一个列变换器：
# transformers=[('encoder', OneHotEncoder(), [3])] 表示对第 3 列（所在州）做独热编码；
# remainder='passthrough' 表示其余各列原样保留、不动
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [3])], remainder='passthrough')
# 在 X 上"学习并执行"这个变换，再用 np.array 把结果转回 numpy 数组
X = np.array(ct.fit_transform(X))

In [6]:
# 打印变换后的 X：原来"州"那一列变成了 3 列 0/1（比如 1,0,0 代表纽约），后面的数值列保持不变
print(X)

[[0.0 0.0 1.0 165349.2 136897.8 471784.1]
 [1.0 0.0 0.0 162597.7 151377.59 443898.53]
 [0.0 1.0 0.0 153441.51 101145.55 407934.54]
 [0.0 0.0 1.0 144372.41 118671.85 383199.62]
 [0.0 1.0 0.0 142107.34 91391.77 366168.42]
 [0.0 0.0 1.0 131876.9 99814.71 362861.36]
 [1.0 0.0 0.0 134615.46 147198.87 127716.82]
 [0.0 1.0 0.0 130298.13 145530.06 323876.68]
 [0.0 0.0 1.0 120542.52 148718.95 311613.29]
 [1.0 0.0 0.0 123334.88 108679.17 304981.62]
 [0.0 1.0 0.0 101913.08 110594.11 229160.95]
 [1.0 0.0 0.0 100671.96 91790.61 249744.55]
 [0.0 1.0 0.0 93863.75 127320.38 249839.44]
 [1.0 0.0 0.0 91992.39 135495.07 252664.93]
 [0.0 1.0 0.0 119943.24 156547.42 256512.92]
 [0.0 0.0 1.0 114523.61 122616.84 261776.23]
 [1.0 0.0 0.0 78013.11 121597.55 264346.06]
 [0.0 0.0 1.0 94657.16 145077.58 282574.31]
 [0.0 1.0 0.0 91749.16 114175.79 294919.57]
 [0.0 0.0 1.0 86419.7 153514.11 0.0]
 [1.0 0.0 0.0 76253.86 113867.3 298664.47]
 [0.0 0.0 1.0 78389.47 153773.43 299737.29]
 [0.0 1.0 0.0 73994.56 122782.75 3

## Splitting the dataset into the Training set and Test set

### 把数据分成训练集和测试集

训练集是"平时练习题"，测试集是"期末考卷"：模型只用训练集学习，再用没见过的测试集检验水平，才能知道它是真学会了还是死记硬背。这里 80% 用来训练、20% 用来测试。

In [7]:
# 从 sklearn 导入"划分数据集"的工具函数
from sklearn.model_selection import train_test_split
# 把数据随机分成训练集（教模型）和测试集（考模型）
# test_size = 0.2 表示 20% 的数据留作测试集；random_state = 42 固定随机种子，保证每次运行分法都一样
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

## Training the Multiple Linear Regression model on the Training set

### 训练模型

fit 就是"训练"：模型会给每个特征自动找一组系数（相当于每个因素的"影响力"），得到类似"利润 = a×研发投入 + b×管理费用 + c×营销费用 + …"的公式，让训练集上的预测误差尽量小。

In [8]:
# 导入 sklearn 中的线性回归模型类
from sklearn.linear_model import LinearRegression
# 创建一个线性回归模型对象（先准备一张"白纸"，等下让它在数据上学习）
regressor = LinearRegression()
# fit 就是"训练"：模型会自动给每个特征分配一个"系数"（影响力大小），使训练集上的预测误差最小
regressor.fit(X_train, y_train)

LinearRegression()

## Predicting the Test set results

### 用模型做预测并对照真实值

对测试集里的公司做利润预测，然后把预测值和真实值并排打印。左右两列数字越接近，说明模型预测得越准。

In [9]:
# 对测试集做预测：输入各公司的研发/管理/营销投入和所在州，模型给出估计的利润 y_pred
y_pred = regressor.predict(X_test)
# 设置打印格式：数字只显示 2 位小数，输出更清爽
np.set_printoptions(precision=2)
# 把"预测利润"和"真实利润"并排拼成一张表打印：
# reshape(len(y),1) 把一维数组竖着排成一列；np.concatenate(..., 1) 表示左右并排拼接
# 左列是模型预测值，右列是真实值，逐行对比就能看出预测准不准
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[126362.88 134307.35]
 [ 84608.45  81005.76]
 [ 99677.49  99937.59]
 [ 46357.46  64926.08]
 [128750.48 125370.37]
 [ 50912.42  35673.41]
 [109741.35 105733.54]
 [100643.24 107404.34]
 [ 97599.28  97427.84]
 [113097.43 122776.86]]
